In [22]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "tensor_engine").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "tensor_engine").is_dir():
    raise RuntimeError("Abre el notebook desde TensorEngine o TensorEngine/notebooks.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# Recarga el código local aunque este kernel haya importado una versión anterior.
for module_name in tuple(sys.modules):
    if module_name == "tensor_engine" or module_name.startswith("tensor_engine."):
        del sys.modules[module_name]

from tensor_engine import (
    AnsatzSpecialization, DimensionSpec, DisplayPolicy, Function,
    LagrangianSourceSpec, Number, ParameterSpec, Scalar,
    TensorEngine, WolframXActBridge, draft4_angular_scalar_profile,
    draft4_circular_ansatz,
)

ansatz = draft4_circular_ansatz()
dimension = DimensionSpec(3)
VALIDAR_XACT = False  # Cambia a True si quieres validar cada caso con xAct.
display_policy = DisplayPolicy(
    factor=True, collect=True, together=True, canonicalize_indices=True,
    aggressive=False, enabled=True, max_nodes=4000,
)

def ejecutar(nombre, lagrangiano, *, ansatz_usado=None, parametros_extra=()):
    parametros = tuple(ParameterSpec(item) for item in parametros_extra)
    if "alpha" in lagrangiano:
        parametros = (ParameterSpec("alpha"), *parametros)
    model = LagrangianSourceSpec(
        name=nombre, expression=lagrangiano, dimension=dimension,
        parameters=parametros,
    ).compile()
    run = TensorEngine().run(
        model, ansatz=ansatz if ansatz_usado is None else ansatz_usado,
        output_root=PROJECT_ROOT / "outputs" / "notebook_cases",
        display_policy=display_policy,
        wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None,
    )
    print(nombre, "|", run.status.value, "|", run.package.verification.summary)
    if run.export_bundle is not None:
        print("Bundle:", run.export_bundle.output_directory)
        if run.export_bundle.pdf_diagnostic:
            print("PDF:", run.export_bundle.pdf_diagnostic)
    return run

In [23]:
# 1. R
run_1 = ejecutar("R", "R")

R | partial | {'passed': 46, 'failed': 0, 'undetermined': 2}
Bundle: C:\Investigacion\TensorEngine\outputs\notebook_cases\r-3a981a5d9948


In [ ]:
# 2. R + alpha R^2
run_2 = ejecutar("R_plus_alpha_times_R_squared", "R + alpha*R**2")

In [ ]:
# 3. R + alpha R_ab R^ab
run_3 = ejecutar("R_plus_alpha_times_RicciSq", "R + alpha*RicciSq")

In [ ]:
# 4. R + alpha R_abcd R^abcd
run_4 = ejecutar("R_plus_alpha_times_RiemannSq", "R + alpha*RiemannSq")

In [ ]:
# 5. R + alpha R_ab nabla^a(phi) nabla^b(phi)
run_5 = ejecutar("R_plus_alpha_times_RicciUU", "R + alpha*RicciUU")

In [ ]:
# 6. R + alpha R X
run_6 = ejecutar("R_plus_alpha_times_RX", "R + alpha*R*X")

In [ ]:
# 7. R + alpha R_abcd nabla^a(phi) nabla^c(phi) nabla^b(phi) nabla^d(phi)
# El término cuártico puede anularse por las antisimetrías de R_abcd.
quartic_riemann_gradient = 'contract(Riemann("a","b","c","d"), metric("a","e"), gradient("e"), metric("c","f"), gradient("f"), metric("b","g"), gradient("g"), metric("d","h"), gradient("h"))'
run_7 = ejecutar("case_07_R_plus_RiemannGrad4", f"R + alpha*{quartic_riemann_gradient}")

In [ ]:
# Especialización opcional posterior: phi = p*varphi.
ansatz_p_varphi = ansatz.specialize_scalar(
    draft4_angular_scalar_profile("p"),
    assumptions=("phi=p*varphi",),
)
run_p_varphi = ejecutar(
    "RicciUU_with_explicit_p_varphi_profile", "R + alpha*RicciUU",
    ansatz_usado=ansatz_p_varphi, parametros_extra=("p",),
)

In [ ]:
# Draft 4 - Caso 0. Edita únicamente f_input y phi_input si quieres probar otra solución.
ansatz_case_0 = draft4_circular_ansatz()
_, r, varphi = ansatz_case_0.chart.coordinates
ell, mass = Scalar("ell"), Scalar("lambda")
f_input = r**2 / ell**2 - mass
phi_input = Number(0)  # El escalar está desacoplado en este caso.
source = LagrangianSourceSpec(name="draft4_case_0", expression="R + 2/ell**2", dimension=DimensionSpec(3), parameters=(ParameterSpec("ell"), ParameterSpec("lambda")))
specialization = AnsatzSpecialization(metric_functions={"f": f_input}, scalar_field=phi_input, name="draft4_case_0_specialized")
run_case_0 = TensorEngine().run(source.compile(), ansatz=ansatz_case_0, specialization=specialization, output_root=PROJECT_ROOT / "outputs" / "draft4_cases", display_policy=display_policy, wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None)
print(run_case_0.export_bundle.output_directory if run_case_0.export_bundle else run_case_0.status.value)

In [ ]:
# Draft 4 - Caso 1. Edita únicamente f_input y phi_input si quieres probar otra solución.
ansatz_case_1 = draft4_circular_ansatz()
_, r, varphi = ansatz_case_1.chart.coordinates
ell, alpha1, p, r0, mass = (Scalar(name) for name in ("ell", "alpha1", "p", "r0", "lambda"))
f_input = r**2 / ell**2 - mass - alpha1*p**2*Function("log", (r/r0,))
phi_input = p*varphi
source = LagrangianSourceSpec(name="draft4_case_1", expression="R + 2/ell**2 - alpha1*X", dimension=DimensionSpec(3), parameters=tuple(ParameterSpec(name) for name in ("ell", "alpha1", "p", "r0", "lambda")))
specialization = AnsatzSpecialization(metric_functions={"f": f_input}, scalar_field=phi_input, name="draft4_case_1_specialized")
run_case_1 = TensorEngine().run(source.compile(), ansatz=ansatz_case_1, specialization=specialization, output_root=PROJECT_ROOT / "outputs" / "draft4_cases", display_policy=display_policy, wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None)
print(run_case_1.export_bundle.output_directory if run_case_1.export_bundle else run_case_1.status.value)

In [ ]:
# Draft 4 - Caso 2. Edita únicamente f_input y phi_input si quieres probar otra solución.
ansatz_case_2 = draft4_circular_ansatz()
_, r, varphi = ansatz_case_2.chart.coordinates
ell, beta0, p, mass = (Scalar(name) for name in ("ell", "beta0", "p", "lambda"))
f_input = (r**2 / ell**2 - mass) / (1 + beta0*p**2*ell**2/r**2)
phi_input = p*varphi
source = LagrangianSourceSpec(name="draft4_case_2", expression="R + 2/ell**2 + ell**2*beta0*(3*RicciUU - X*R)", dimension=DimensionSpec(3), parameters=tuple(ParameterSpec(name) for name in ("ell", "beta0", "p", "lambda")))
specialization = AnsatzSpecialization(metric_functions={"f": f_input}, scalar_field=phi_input, name="draft4_case_2_specialized")
run_case_2 = TensorEngine().run(source.compile(), ansatz=ansatz_case_2, specialization=specialization, output_root=PROJECT_ROOT / "outputs" / "draft4_cases", display_policy=display_policy, wolfram_bridge=WolframXActBridge(timeout_seconds=300) if VALIDAR_XACT else None)
print(run_case_2.export_bundle.output_directory if run_case_2.export_bundle else run_case_2.status.value)